# Using OCOD prices vs using the Price Paid dataset

One may ask why use the price paid dataset instead of the the prices that are already in the OCOD collection

In [ ]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
from plotnine import *
import seaborn as sns
from enhance_ocod.analysis import create_time_series_by_groups, create_mean_difference_by_groups
from enhance_ocod.address_parsing import load_csv_from_zip

data_folder = Path('../data') 
figures_folder = Path('../figures/figures')

raw_ocod_folder = data_folder /'ocod_history'
processed_ocod_folder = data_folder /'ocod_history_processed'
target_ocod_file = 'OCOD_FULL_2022_02'

price_paid_msoa_average = data_folder / 'price_paid_msoa_averages' /'price_paid_2022_02.parquet'



In [ ]:
# load the msoa price data
price_paid_msoa_average = pd.read_parquet(price_paid_msoa_average)

In [ ]:
# load the ocod titles with the ocod price data
OCOD_raw_df = load_csv_from_zip(
                    zip_path=raw_ocod_folder /(target_ocod_file+'.zip'),
                    csv_filename='OCOD_FULL_2022_02.csv',
                    encoding_errors="ignore",
                )

OCOD_raw_df = OCOD_raw_df.rename(columns = {'Title Number':'title_number', 'Price Paid':'ocod_price'})
OCOD_raw_df = OCOD_raw_df[['title_number', 'ocod_price']]
OCOD_raw_df['no_ocod_price'] = OCOD_raw_df['ocod_price'].isna()

In [ ]:
# load the processed ocod data

OCOD_processed_df = pd.read_parquet(processed_ocod_folder/(target_ocod_file+'.parquet'))

OCOD_processed_df = OCOD_processed_df.loc[OCOD_processed_df['class']=='residential']

In [ ]:
# add in price data

OCOD_processed_df = OCOD_processed_df.merge(price_paid_msoa_average,  on  = 'msoa11cd').merge(OCOD_raw_df, on = 'title_number')


In [ ]:
OCOD_raw_df.groupby('no_ocod_price').size()/OCOD_processed_df.shape[0]

In [ ]:
OCOD_processed_df.groupby('no_ocod_price').size()/OCOD_processed_df.shape[0]

## Comparison of price distribution

It is clear from the plot below that the OCOD price has a higher mean but that the median value is quite similar. A closer inspection of the upper end of the OCOD prices shows that 94\% were multi-properties, of the bottom end 28\% were multi-properties.

In [ ]:
OCOD_processed_df[['price_mean', 'price_median','ocod_price']].describe()

In [ ]:
top_group =  OCOD_processed_df.loc[~OCOD_processed_df['no_ocod_price']].sort_values('ocod_price').tail(100)

top_group[['is_multi']].describe()


In [ ]:
bottom_group = OCOD_processed_df.loc[~OCOD_processed_df['no_ocod_price']].sort_values('ocod_price').head(100)

bottom_group[['is_multi']].describe()

In [ ]:
bottom_group[['price_mean', 'price_median','ocod_price']].describe()

In [ ]:
ggplot(bottom_group,
aes(x = 'expansion_size')) + geom_histogram()

In [ ]:
ggplot(bottom_group, aes(x = 'ocod_price', y = 'price_median')) +geom_point()

In [ ]:
ggplot(top_group, aes(x = 'ocod_price', y = 'price_median')) +geom_point()

In [ ]:
ggplot(OCOD_processed_df, aes(x = 'ocod_price', y = 'price_median')) +geom_point()

In [ ]:
MSOA_totals = OCOD_processed_df.groupby('msoa11cd')[['price_mean', 'price_median', 'ocod_price']].sum()

ggplot(MSOA_totals, aes(x = 'price_median', y = 'ocod_price')) + geom_point() 

In [ ]:
MSOA_totals.sum()